# Lab 2B: Thinking and effort
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 2 · about 20 minutes**

**Objectives**
- Compare answers with and without adaptive thinking, checked in code
- Measure how effort changes tokens and time
- Inspect thinking blocks and usage
- Handle max_tokens and model support when thinking is on

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## Drill 1: thinking on or off
A counting problem with an answer we can compute in Python.

In [ ]:
PROBLEM = ("A telecom assigns support ticket numbers from 1 to 5000. A ticket is flagged if its number is divisible by 7 or by 11, "
           "but not by both. How many tickets are flagged? End your reply with a line: ANSWER: <number>")
truth = sum(1 for k in range(1, 5001) if (k % 7 == 0) != (k % 11 == 0))
print("ground truth:", truth)

def answer_of(resp):
    m = re.search(r"ANSWER:\s*([\d,]+)", text_of(resp))
    return int(m.group(1).replace(",", "")) if m else None

def run(label, **kw):
    t0 = time.perf_counter()
    r = client.messages.create(model=config.MODEL, max_tokens=kw.pop("max_tokens", 8000),
                               messages=[{"role": "user", "content": PROBLEM}], **kw)
    dt = time.perf_counter() - t0
    a = answer_of(r)
    print(f"{label:22s} answer={a} correct={a == truth} output_tokens={r.usage.output_tokens:5d} time={dt:5.1f}s blocks={[b.type for b in r.content]}")
    return r

no_think = run("no thinking")
adaptive = run("adaptive, default", thinking={"type": "adaptive"})

## Drill 2: effort levels
**Predict:** which effort level uses the fewest output tokens? Does accuracy change on this problem?

In [ ]:
results = {}
for effort in ["low", "medium", "high"]:
    results[effort] = run(f"adaptive, effort={effort}", thinking={"type": "adaptive"}, output_config={"effort": effort})

Thinking tokens are billed as output tokens. Pick the lowest effort that keeps quality on **your** evaluation set.

## Drill 3: read the blocks

In [ ]:
r = results["high"]
for b in r.content:
    if b.type == "thinking":
        print("THINKING (summary shown to you):", b.thinking[:500].replace("\n", " "), "...")
        print("signature present:", bool(getattr(b, "signature", None)))
    elif b.type == "text":
        print("\nTEXT:", b.text[-300:])
print("\nusage:", r.usage)

Two rules to remember:
- You see a **summary** of the thinking, but you are billed for the full thinking.
- In tool loops (Day 3), pass thinking blocks back **unchanged**, signature included, with the assistant turn.

## Drill 4: limits and model support

In [ ]:
# 4a. max_tokens covers thinking AND the answer
tight = client.messages.create(model=config.MODEL, max_tokens=300, thinking={"type": "adaptive"},
                               output_config={"effort": "high"}, messages=[{"role": "user", "content": PROBLEM}])
print("tight budget -> stop_reason:", tight.stop_reason, "| blocks:", [b.type for b in tight.content])

In [ ]:
# 4b. Which thinking mode does the fast model accept? Predict first.
try:
    client.messages.create(model=config.FAST_MODEL, max_tokens=4000, thinking={"type": "adaptive"},
                           messages=[{"role": "user", "content": PROBLEM}])
    print("adaptive accepted on", config.FAST_MODEL)
except anthropic.BadRequestError as e:
    print("adaptive rejected on", config.FAST_MODEL, "->", str(e)[:160])

manual = client.messages.create(model=config.FAST_MODEL, max_tokens=6000,
                                thinking={"type": "enabled", "budget_tokens": 4000},
                                messages=[{"role": "user", "content": PROBLEM}])
a = answer_of(manual)
print(f"manual budget on {config.FAST_MODEL}: answer={a} correct={a == truth} output_tokens={manual.usage.output_tokens}")

The right thinking mode depends on the model. Keep it next to the model ID in `config.py`, so a model change and its thinking settings change together.

## You should now have
- [ ] A with and without thinking comparison, checked in code
- [ ] Token and time figures for three effort levels
- [ ] An example of thinking exhausting max_tokens
- [ ] The thinking mode each of your two models accepts

## Check yourself
1. What does the effort setting control?
2. Why can a response with thinking end with no text block?
3. What must you do with thinking blocks in a tool loop?